# Model Training

## Import necessary libraries

In [1]:
%pip install -qq -r ../requirements.txt

Note: you may need to restart the kernel to use updated packages.


In [2]:
# Add current directory to Python path for imports
import os
import sys

# Add the parent directory (project root) to Python path so we can import from src
project_root = os.path.dirname(os.getcwd())
if project_root not in sys.path:
    sys.path.append(project_root)

In [3]:
# Utility Functions
from src.utils import create_spark_session

# Create Spark session
spark, sedona = create_spark_session(app_name="ModelTrainingSpark")

## Loading Datasets

In [4]:
from src.utils import read_config_path

# Load data using configuration file
filepath = read_config_path(key="model_training_data_path", domain="processed")

df_raw = spark.read.csv(
    filepath,
    header=True,
    inferSchema=True,
    multiLine=True,
    escape='"',
    quote='"',
)

In [5]:
from src.utils import preprocessed_data_converter

df_prepared = preprocessed_data_converter(df_raw)

In [6]:
df_prepared.show(5, truncate=False)

+---------------+--------------+---------------+----------------------------+--------------------+-------------------------+--------------------+
|timestamp_month|timestamp_year|resolution_time|address_encoded             |latlong_encoded     |organization_encoded     |type_encoded        |
+---------------+--------------+---------------+----------------------------+--------------------+-------------------------+--------------------+
|9              |2021          |275            |(2048,[834,1804],[1.0,1.0]) |[13.67891,100.66709]|(1786,[10,53],[1.0,1.0]) |(25,[7,8],[1.0,1.0])|
|9              |2021          |253            |(2048,[348,426],[1.0,1.0])  |[13.7206,100.52649] |(1786,[49],[1.0])        |(25,[14],[1.0])     |
|12             |2021          |246            |(2048,[802,1656],[1.0,1.0]) |[13.8228,100.59165] |(1786,[31,108],[1.0,1.0])|(25,[0,8],[1.0,1.0])|
|12             |2021          |456            |(2048,[802,1656],[1.0,1.0]) |[13.8091,100.59131] |(1786,[31,172],[1.0,1.0])|

In [7]:
df_prepared.printSchema()

root
 |-- timestamp_month: integer (nullable = true)
 |-- timestamp_year: integer (nullable = true)
 |-- resolution_time: integer (nullable = true)
 |-- address_encoded: vector (nullable = true)
 |-- latlong_encoded: vector (nullable = true)
 |-- organization_encoded: vector (nullable = true)
 |-- type_encoded: vector (nullable = true)



---

## Model Training

### Sampling the preprocessed data

In [8]:
from src.utils import preprocessed_data_sampler

train_df, test_df = preprocessed_data_sampler(df_prepared, sampling_fraction=0.1)

---

### Gradient Boosted Tree Regressor Model

In [9]:
from pyspark.ml.regression import GBTRegressor

from src.pipelines_spark import ModelDefinePipelineSpark


gbt = GBTRegressor(
    labelCol="resolution_time",
    featuresCol="features",
)

gbt_model_pipeline = ModelDefinePipelineSpark(
    name="GradientBoostedTree",
    model=gbt,
    input_columns=[
        "timestamp_month",
        "timestamp_year",
        "address_encoded",
        "latlong_encoded",
        "organization_encoded",
        "type_encoded",
    ],
    label_column="resolution_time",
    evaluators=["rmse", "mae", "r2"],
    param_dict={
        "maxDepth": [3, 4, 5, 6],
        "maxIter": [50, 80, 100],
        "stepSize": [0.05, 0.1],
        "subsamplingRate": [0.8],
    },
)

In [ ]:
gbt_model_pipeline.fit(train_df, save_name="gbt")

In [10]:
from pyspark.ml.tuning import CrossValidatorModel

from src.utils import get_model_path

model_path = str(get_model_path("gbt"))

model = CrossValidatorModel.load(model_path)

gbt_model_pipeline.set_model(model)
gbt_score = gbt_model_pipeline.evaluate(test_df)


===== GradientBoostedTree Results =====
RMSE: 86.51564341480632
MAE: 49.14561449093745
R2: 0.5765039934534153


In [11]:
best_gbt_params = gbt_model_pipeline.get_best_params()

maxDepth = 6
maxIter = 100
stepSize = 0.1
subsamplingRate = 0.8


---

### Random Forest Regressor Model

In [12]:
from pyspark.ml.regression import RandomForestRegressor

from src.pipelines_spark import ModelDefinePipelineSpark


rf = RandomForestRegressor(
    labelCol="resolution_time",
    featuresCol="features",
)

rf_model_pipeline = ModelDefinePipelineSpark(
    name="RandomForestRegressor",
    model=rf,
    input_columns=[
        "timestamp_month",
        "timestamp_year",
        "address_encoded",
        "latlong_encoded",
        "organization_encoded",
        "type_encoded",
    ],
    label_column="resolution_time",
    evaluators=["rmse", "mae", "r2"],
    param_dict={
        "numTrees": [50, 80, 100, 120, 150, 200],
        "maxDepth": [6, 8, 10, 12],
    },
)

In [ ]:
rf_model_pipeline.fit(train_df, save_name="rf")

In [13]:
from pyspark.ml.tuning import CrossValidatorModel

from src.utils import get_model_path

model_path = str(get_model_path("rf"))

model = CrossValidatorModel.load(model_path)

rf_model_pipeline.set_model(model)
rf_score = rf_model_pipeline.evaluate(test_df)


===== RandomForestRegressor Results =====
RMSE: 91.56537700828464
MAE: 53.946658585692845
R2: 0.5256241064818494


In [14]:
best_rf_params = rf_model_pipeline.get_best_params()

maxDepth = 12
numTrees = 120


---

## Train Best Model on Full Training Data

### Get Best Model and Parameters

In [15]:
if gbt_score["rmse"] < rf_score["rmse"]:
    print("GBT model performs better.")
    print(f"Best GBT Params: {best_gbt_params}")

    best_model = GBTRegressor(
        labelCol="resolution_time",
        featuresCol="features",
    ).setParams(**best_gbt_params)

else:
    print("RF model performs better.")
    print(f"Best RF Params: {best_rf_params}")

    best_model = RandomForestRegressor(
        labelCol="resolution_time",
        featuresCol="features",
    ).setParams(**best_rf_params)

GBT model performs better.
Best GBT Params: {'maxDepth': 6, 'maxIter': 100, 'stepSize': 0.1, 'subsamplingRate': 0.8}


---

### Prepare Full Training Data

In [16]:
train_df, test_df = preprocessed_data_sampler(df_prepared, sampling_fraction=1.0)

---

### Setting up the Best Model

In [17]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler

assembler = VectorAssembler(
    inputCols=[
        "timestamp_month",
        "timestamp_year",
        "address_encoded",
        "latlong_encoded",
        "organization_encoded",
        "type_encoded",
    ],
    outputCol="features",
)

best_model_pipeline = Pipeline(stages=[assembler, best_model])

In [ ]:
best_model_pipeline.fit(train_df)

In [ ]:
save_path = str(get_model_path("best_model_full_data"))

best_model_pipeline.save(save_path)

---

### Test the best hyperparameters on full training data

In [ ]:
from typing import cast
from pyspark.ml.evaluation import RegressionEvaluator

from src.utils import evaluate_model, get_model_path

model_path = str(get_model_path("best_model_full_data"))

best_model_pipeline = Pipeline.load(model_path)

evaluators = ["rmse", "mae", "r2"]
evaluators_dict = {}

for evaluator in evaluators:
    evaluators_dict[evaluator] = RegressionEvaluator(
        labelCol="resolution_time",
        predictionCol="prediction",
        metricName=evaluator,  # type: ignore
    )

evaluate_model(
    name="Best Model on Full Data",
    model=best_model_pipeline,
    test_df=test_df,
    evaluators=evaluators_dict,
)


===== Best Model on Full Data Results =====


AttributeError: 'GBTRegressor' object has no attribute 'transform'

---

## Stop Spark

In [ ]:
spark.stop()

---